In [7]:
import yfinance as yf

# 1. 미국 시장(US) 객체 생성
us_market = yf.Market("US")

# 미국 시장의 현재 개장 상태 확인
print(f"미국 시장 상태: {us_market.status}")

# 미국 3대 주요 지수(S&P500, 나스닥 등) 요약 정보 확인
import pandas as pd
display(pd.DataFrame(us_market.summary).T.head())

# ==========================================

# 2. 원자재 시장(COMMODITIES) 객체 생성
commodities = yf.Market("COMMODITIES")

# 원자재 가격 요약 (금, 은, 원유 등)
print("\n--- 주요 원자재 가격 동향 ---")
display(pd.DataFrame(commodities.summary).T.head())

미국 시장 상태: {'id': 'us', 'name': 'U.S. markets', 'status': 'closed', 'yfit_market_id': 'us_market', 'close': datetime.datetime(2026, 4, 13, 20, 0, tzinfo=datetime.timezone.utc), 'duration': [{'hrs': '1', 'mins': '58'}], 'message': 'U.S. markets open in 1 hour 58 minutes', 'open': datetime.datetime(2026, 4, 13, 13, 30, tzinfo=datetime.timezone.utc), 'yfit_market_status': 'YFT_MARKET_WILL_OPEN', 'timezone': {'dst': 'true', 'gmtoffset': '-14400', 'short': 'EDT', '$text': 'America/New_York'}, 'tz': datetime.timezone(datetime.timedelta(days=-1, seconds=34560), 'EDT')}


,language,region,quoteType,typeDisp,quoteSourceName,triggerable,customPriceAlertConfidence,contractSymbol,headSymbolAsString,shortName,...,exchangeTimezoneName,exchangeTimezoneShortName,gmtOffSetMilliseconds,esgPopulated,tradeable,cryptoTradeable,hasPrePostMarketData,firstTradeDateMilliseconds,symbol,priceHint
CME,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,False,RTY=F,Russell 2000 Futures,...,America/New_York,EDT,-14400000,False,False,False,False,1499659200000,RTY=F,NaN
CBT,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,False,YM=F,Dow Futures,...,America/New_York,EDT,-14400000,False,False,False,False,1017982800000,YM=F,NaN
CXI,en-US,US,INDEX,Index,NaN,False,LOW,NaN,NaN,VIX,...,America/Chicago,CDT,-18000000,False,False,False,False,631267200000,^VIX,2
CMX,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,False,GC=F,Gold,...,America/New_York,EDT,-14400000,False,False,False,False,967608000000,GC=F,NaN



--- 주요 원자재 가격 동향 ---


,language,region,quoteType,typeDisp,quoteSourceName,triggerable,customPriceAlertConfidence,headSymbolAsString,contractSymbol,shortName,...,hasPrePostMarketData,firstTradeDateMilliseconds,regularMarketChange,regularMarketChangePercent,regularMarketTime,regularMarketPrice,regularMarketPreviousClose,exchange,market,symbol
NYM,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,BZ=F,False,Brent Crude Oil Last Day Financ,...,False,1185768000000,6.75,7.090336,1776079382,101.95,95.2,NYM,us24_market,BZ=F
CMX,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,HG=F,False,Copper May 26,...,False,967608000000,-0.0155,-0.263338,1776079395,5.8705,5.886,CMX,us24_market,HG=F


In [8]:
import yfinance as yf
from datetime import datetime, timedelta

# Default init (today + 7 days)
calendar = yf.Calendars()

# Today's events: calendar of 1 day
tomorrow = datetime.now() + timedelta(days=1)
calendar = yf.Calendars(end=tomorrow)

# Default calendar queries - accessing the properties will fetch the data from YF
calendar.earnings_calendar
calendar.ipo_info_calendar
calendar.splits_calendar
calendar.economic_events_calendar

# Manual queries
calendar.get_earnings_calendar()
calendar.get_ipo_info_calendar()
calendar.get_splits_calendar()
calendar.get_economic_events_calendar()

# Earnings calendar custom filters
calendar.get_earnings_calendar(
    market_cap=100_000_000,  # filter out small-cap 
    filter_most_active=True,  # show only actively traded. Uses: `screen(query="MOST_ACTIVES")`
)

# Example of real use case:
# Get inminent unreported earnings events
today = datetime.now()
is_friday = today.weekday() == 4
day_after_tomorrow = today + timedelta(days=4 if is_friday else 2)

calendar = yf.Calendars(today, day_after_tomorrow)
df = calendar.get_earnings_calendar(limit=100)

unreported_df = df[df["Reported EPS"].isnull()]

In [9]:
import yfinance as yf

# 테슬라(TSLA) 객체 생성
tsla = yf.Ticker("TSLA")

# 1. 애널리스트 목표주가 확인 (딕셔너리로 반환됨)
targets = tsla.analyst_price_targets
print("--- 테슬라 애널리스트 목표주가 ---")
print(f"최고 목표가: ${targets.get('high')}")
print(f"평균 목표가: ${targets.get('mean')}")
print(f"최저 목표가: ${targets.get('low')}")

# 2. 다음 분기 및 내년 실적 예상치 (데이터프레임)
# .loc[]를 써서 올해(0y)와 내년(+1y) 예측치만 필터링해 볼 수 있습니다.
estimates = tsla.earnings_estimate
print("\n--- 테슬라 EPS 실적 예상치 ---")
print(estimates)

# 3. 최대 기관 투자자 목록 확인
institutions = tsla.institutional_holders
print("\n--- 주요 기관 투자자 ---")
# 기관 이름(Holder)과 보유 주식 수(Shares)만 추출
print(institutions[['Holder', 'Shares']].head())

--- 테슬라 애널리스트 목표주가 ---
최고 목표가: $600.0
평균 목표가: $415.2966
최저 목표가: $125.0

--- 테슬라 EPS 실적 예상치 ---
            avg      low  high  yearAgoEps  numberOfAnalysts  growth
period                                                              
0q      0.37735  0.21073  0.51     0.27000                25  0.3976
+1q     0.44637  0.22807  0.59     0.40000                24  0.1159
0y      2.04255  1.11582  2.82     1.66000                34  0.2305
+1y     2.77327  1.43809  6.03     2.04255                33  0.3577

--- 주요 기관 투자자 ---
                          Holder     Shares
0             Vanguard Group Inc  258925024
1                 Blackrock Inc.  209563808
2       State Street Corporation  114842934
3  Geode Capital Management, LLC   65700975
4            JPMORGAN CHASE & CO   44591616


In [10]:
import yfinance as yf

# 1. 의도적으로 오타가 섞인 기업명으로 검색 (Nvidya -> Nvidia)
# enable_fuzzy_query=True를 켜서 유사 검색을 허용합니다.
search_result = yf.Search("Nvidya", enable_fuzzy_query=True, max_results=3)

# 2. 검색된 종목(Quotes) 결과 확인
quotes = search_result.quotes

print("--- 종목 검색 결과 ---")
for quote in quotes:
    # 딕셔너리에서 심볼(티커)과 공식 회사명, 주식 거래소(Exchange) 추출
    symbol = quote.get('symbol')
    short_name = quote.get('shortname')
    exchange = quote.get('exchange')
    print(f"티커: {symbol} | 기업명: {short_name} | 거래소: {exchange}")

# 3. 찾은 티커 중 첫 번째(가장 정확한 결과)를 이용하여 Ticker 객체 생성 후 가격 확인
if quotes:
    exact_ticker = quotes[0].get('symbol') # 'NVDA'가 추출됨
    
    # 여기서부터는 이전에 배운 Ticker 클래스의 영역입니다!
    nvda = yf.Ticker(exact_ticker)
    current_price = nvda.fast_info.last_price
    print(f"\n=> 검색된 {exact_ticker}의 현재가: ${current_price:.2f}")

# 4. 관련된 최신 뉴스 확인
print("\n--- 관련 최신 뉴스 ---")
news_list = search_result.news
for article in news_list[:2]: # 2개만 출력
    print(f"- {article.get('title')}")

--- 종목 검색 결과 ---
티커: NVDA | 기업명: NVIDIA Corporation | 거래소: NMS
티커: NVDY | 기업명: YieldMax NVDA Option Income Str | 거래소: PCX
티커: NVDU | 기업명: Direxion Daily NVDA Bull 2X ETF | 거래소: NGM

=> 검색된 NVDA의 현재가: $188.63

--- 관련 최신 뉴스 ---
- With Brain Scan Volume Momentum Accelerating, Firefly Makes History by Building the World’s First Known 200,000+ Standardized EEG/ERP Depository
- Ninepoint Partners Launching Expanded Suite of Single-Stock ETFs


In [17]:
import yfinance as yf
import pandas as pd

# 1. 기존의 Search 객체를 통해 모든 결과 검색
search_result = yf.Search("Gold", max_results=30)
df_all = pd.DataFrame(search_result.quotes)

# 2. 'quoteType' 컬럼을 검사하여 'EQUITY'(개별 주식)만 필터링
if not df_all.empty and 'quoteType' in df_all.columns:
    df_stocks = df_all[df_all['quoteType'] == 'EQUITY']
    print("--- 직접 필터링한 주식 목록 ---")
    print(df_stocks[['symbol', 'shortname', 'exchange']])
else:
    print("검색 결과가 없습니다.")

--- 직접 필터링한 주식 목록 ---
  symbol       shortname exchange
0   GOLD  Gold.com, Inc.      NYQ
